# Blue Book Encounter Phenomenology: OCR & NLP Pipeline

Colab Pro optimized. Run on GPU runtime (A100 preferred, T4 workable).

**Pipeline:**
1. Download Blue Book PDFs from Internet Archive (decade ZIPs)
2. OCR with Marker (files already processed) + Google Cloud Vision (remaining files)
3. Clean OCR output (strip form boilerplate, noise, artifacts)
4. Domain-aware spell correction
5. Embed full documents with nomic-embed-text-v1.5 (8192 token context)
6. UMAP + HDBSCAN clustering
7. Visualization and analysis

All data persists on Google Drive across sessions.

## Cell 1: Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import subprocess
import json

PROJECT_DIR = "/content/drive/MyDrive/blue_book_phenomenology"
CASE_DIR = f"{PROJECT_DIR}/cases"
MARKER_OUT_DIR = f"{PROJECT_DIR}/marker_output"
GCV_OUT_DIR = f"{PROJECT_DIR}/gcv_output"
CLEAN_DIR = f"{PROJECT_DIR}/corpus_clean"
CORRECTED_DIR = f"{PROJECT_DIR}/corpus_corrected"
FINAL_CORPUS_DIR = f"{PROJECT_DIR}/corpus"
METADATA_DIR = f"{PROJECT_DIR}/metadata"
RESULTS_DIR = f"{PROJECT_DIR}/results"
DATA_DIR = f"{RESULTS_DIR}/data"
MODELS_DIR = f"{RESULTS_DIR}/models"
VIZ_DIR = f"{RESULTS_DIR}/visualizations"
DOCS_DIR = f"{PROJECT_DIR}/docs"

for d in [PROJECT_DIR, CASE_DIR, MARKER_OUT_DIR, GCV_OUT_DIR,
          CLEAN_DIR, CORRECTED_DIR, FINAL_CORPUS_DIR, METADATA_DIR,
          RESULTS_DIR, DATA_DIR, MODELS_DIR, VIZ_DIR, DOCS_DIR]:
    os.makedirs(d, exist_ok=True)

print("Project directory structure ready.")


## Cell 2: Install Dependencies

In [ ]:
%%bash
pip install marker-pdf -q
pip install google-cloud-vision -q
pip install PyMuPDF -q
pip install symspellpy -q
pip install sentence-transformers einops -q
pip install umap-learn hdbscan bertopic -q
pip install polars pyarrow tqdm requests -q

echo "Dependencies installed."

## Cell 3: GPU Detection

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Switch to GPU runtime.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

## Cell 4: Brad Sparks Unknowns Catalog

The 701 Unidentified cases list. Download manually and place at:
`/content/drive/MyDrive/blue_book_phenomenology/metadata/bb_unknowns.csv`

See: https://www.nicap.org/bb/BB_Unknowns.pdf

In [ ]:
import polars as pl

unknowns_path = f"{METADATA_DIR}/bb_unknowns.csv"

if os.path.exists(unknowns_path):
    unknowns = pl.read_csv(unknowns_path)
    print(f"Loaded {len(unknowns)} Unidentified cases from index.")
else:
    print("WARNING: Unidentified cases index not found.")
    print(f"Place Brad Sparks' catalog at: {unknowns_path}")

## Cell 5: Download Case Files from Internet Archive

| File | Size | Coverage |
|------|------|----------|
| `1940s.zip` | 1.0 GB | 505 cases (1947-1949) |
| `1950s.zip` | 7.2 GB | ~4,000 cases |
| `1960s.zip` | 12.3 GB | ~5,500 cases |
| `19XXs.zip` | 651 MB | Undated/miscellaneous |

**Total:** ~21 GB, pre-segmented into individual case files.

Source: https://archive.org/details/bluebook

In [ ]:
import requests
import zipfile
from tqdm import tqdm

ARCHIVE_BASE = "https://archive.org/download/bluebook"
DECADE_ZIPS = ["1940s.zip", "1950s.zip", "1960s.zip", "19XXs.zip"]

def download_and_extract_decade(zip_name, output_dir):
    decade = zip_name.replace(".zip", "")
    decade_dir = os.path.join(output_dir, decade)
    if os.path.exists(decade_dir) and len(os.listdir(decade_dir)) > 0:
        count = sum(1 for r, d, f in os.walk(decade_dir) for x in f if x.lower().endswith('.pdf'))
        print(f"  {decade}: already extracted ({count} case files)")
        return decade_dir
    os.makedirs(decade_dir, exist_ok=True)
    url = f"{ARCHIVE_BASE}/{zip_name}"
    print(f"  Downloading {zip_name}...")
    resp = requests.get(url, stream=True, timeout=300)
    resp.raise_for_status()
    total_size = int(resp.headers.get('content-length', 0))
    temp_path = os.path.join(output_dir, f"_temp_{zip_name}")
    with open(temp_path, 'wb') as f:
        with tqdm(total=total_size, unit='B', unit_scale=True, desc=decade, leave=False) as pbar:
            for chunk in resp.iter_content(chunk_size=131072):
                f.write(chunk)
                pbar.update(len(chunk))
    print(f"  Extracting {zip_name}...")
    with zipfile.ZipFile(temp_path, 'r') as zf:
        zf.extractall(decade_dir)
    os.remove(temp_path)
    count = sum(1 for r, d, f in os.walk(decade_dir) for x in f if x.lower().endswith('.pdf'))
    print(f"  {decade}: extracted {count} case files")
    return decade_dir

print("Downloading complete Blue Book archive from Internet Archive...")
print("(~21 GB total — this will take a while on first run)\n")
case_dirs = []
for zip_name in DECADE_ZIPS:
    try:
        result = download_and_extract_decade(zip_name, CASE_DIR)
        case_dirs.append(result)
    except Exception as e:
        print(f"  ERROR downloading {zip_name}: {e}")

import glob
all_pdfs = glob.glob(f"{CASE_DIR}/**/*.pdf", recursive=True) + \
           glob.glob(f"{CASE_DIR}/**/*.PDF", recursive=True)
print(f"\nTotal case files: {len(all_pdfs)}")

## Cell 6: Marker OCR (Local SSD + Checkpointing)

Copies PDFs to local SSD, runs Marker with `force_ocr=True` in batches of 20. Each batch checkpoints to Drive. Skips already-processed files on resume. ~1,205 files already done from prior runs.

In [ ]:
import shutil
import subprocess
import glob

LOCAL_PDF_DIR = "/content/cases_local"
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)

print("Copying PDFs from Drive to local SSD (bulk)...")
for decade in ["1940s", "1950s", "1960s", "19XXs"]:
    src = os.path.join(CASE_DIR, decade)
    if os.path.exists(src):
        print(f"  Copying {decade}...")
        subprocess.run(["cp", "-rn", src, LOCAL_PDF_DIR], check=True)
print("Copy complete.")

local_pdfs = glob.glob(f"{LOCAL_PDF_DIR}/**/*.pdf", recursive=True) + \
             glob.glob(f"{LOCAL_PDF_DIR}/**/*.PDF", recursive=True)
print(f"Total PDFs on local SSD: {len(local_pdfs)}")

to_process = []
skipped = 0
for pdf_path in local_pdfs:
    basename = os.path.splitext(os.path.basename(pdf_path))[0]
    drive_md = os.path.join(MARKER_OUT_DIR, f"{basename}.md")
    if os.path.exists(drive_md) and os.path.getsize(drive_md) > 0:
        skipped += 1
    else:
        to_process.append(pdf_path)

print(f"Already processed: {skipped}")
print(f"Remaining: {len(to_process)}")

if not to_process:
    print("All files already processed!")
else:
    BATCH_SIZE = 20
    N_WORKERS = 2
    batches = [to_process[i:i+BATCH_SIZE] for i in range(0, len(to_process), BATCH_SIZE)]
    print(f"\nProcessing {len(to_process)} files in {len(batches)} batches of {BATCH_SIZE}")
    print(f"Using {N_WORKERS} parallel workers\n")

    for batch_idx, batch in enumerate(batches):
        batch_input = "/content/_batch_input"
        batch_output = "/content/_batch_output"
        if os.path.exists(batch_input): shutil.rmtree(batch_input)
        if os.path.exists(batch_output): shutil.rmtree(batch_output)
        os.makedirs(batch_input)
        os.makedirs(batch_output)

        for pdf_path in batch:
            link_path = os.path.join(batch_input, os.path.basename(pdf_path))
            if not os.path.exists(link_path):
                os.symlink(pdf_path, link_path)

        print(f"Batch {batch_idx + 1}/{len(batches)}: {len(batch)} files...")
        result = subprocess.run(
            ["marker", batch_input,
             "--output_dir", batch_output,
             "--workers", str(N_WORKERS),
             "--force_ocr",
             "--disable_image_extraction",
             "--skip_existing"],
            capture_output=False,
        )

        copied = 0
        for root, dirs, files in os.walk(batch_output):
            for f in files:
                if f.endswith('.md'):
                    src = os.path.join(root, f)
                    shutil.copy2(src, os.path.join(MARKER_OUT_DIR, f))
                    copied += 1

        shutil.rmtree(batch_input)
        shutil.rmtree(batch_output)
        total_done = skipped + (batch_idx + 1) * BATCH_SIZE
        print(f"  Batch {batch_idx + 1} done: {copied} files -> Drive "
              f"({min(total_done, len(local_pdfs))}/{len(local_pdfs)} total)\n")

    print("All batches complete.")

## Cell 7: Google Cloud Vision OCR via Cloud Storage (Batch)

Uploads remaining PDFs to a GCS bucket, runs GCV async batch API server-side in parallel, downloads text results to Drive. No local rendering, no GPU needed.

**Steps:**
1. Upload PDFs from Drive to GCS bucket (Google-to-Google, fast)
2. Submit async batch OCR requests (100 files per batch, processed in parallel)
3. Wait for all batches to complete
4. Download text results back to Drive as `.md` files

**Cost:** ~$1.50 per 1,000 pages OCR + ~$0.50 GCS storage (temporary). Estimated ~$207 total.

**Requirements:** GCP project `blue-book-ocr` with Cloud Vision API enabled and billing attached.


In [ ]:
!pip install google-cloud-vision google-cloud-storage -q

import os
import json
import glob
import re
from tqdm import tqdm
from google.cloud import vision, storage
from google.colab import auth

# --- Auth ---
auth.authenticate_user()
import google.auth
credentials, _ = google.auth.default()

PROJECT_ID = "blue-book-ocr"
BUCKET_NAME = "blue-book-ocr-pdfs"
GCS_INPUT_PREFIX = "pdfs/"
GCS_OUTPUT_PREFIX = "ocr_results/"

# --- Create bucket if needed ---
storage_client = storage.Client(project=PROJECT_ID, credentials=credentials)
try:
    bucket = storage_client.get_bucket(BUCKET_NAME)
    print(f"Bucket exists: {BUCKET_NAME}")
except:
    bucket = storage_client.create_bucket(BUCKET_NAME, location="us-central1")
    print(f"Created bucket: {BUCKET_NAME}")

# --- Find files needing OCR ---
all_pdfs = glob.glob(f"{CASE_DIR}/**/*.pdf", recursive=True) + \
           glob.glob(f"{CASE_DIR}/**/*.PDF", recursive=True)

to_process = []
skipped = 0
for pdf_path in all_pdfs:
    basename = os.path.splitext(os.path.basename(pdf_path))[0]
    if os.path.exists(os.path.join(MARKER_OUT_DIR, f"{basename}.md")):
        skipped += 1; continue
    if os.path.exists(os.path.join(GCV_OUT_DIR, f"{basename}.md")):
        skipped += 1; continue
    to_process.append(pdf_path)

print(f"Total PDFs: {len(all_pdfs)}")
print(f"Already done: {skipped}")
print(f"Remaining: {len(to_process)}")

# --- Step 1: Upload PDFs to GCS ---
print(f"\nUploading {len(to_process)} PDFs to gs://{BUCKET_NAME}/{GCS_INPUT_PREFIX}...")
uploaded = 0
for pdf_path in tqdm(to_process, desc="Uploading"):
    basename = os.path.basename(pdf_path)
    blob = bucket.blob(f"{GCS_INPUT_PREFIX}{basename}")
    if blob.exists():
        uploaded += 1; continue
    blob.upload_from_filename(pdf_path)
    uploaded += 1
print(f"Uploaded: {uploaded}")

# --- Step 2: Run async batch OCR ---
vision_client = vision.ImageAnnotatorClient(
    credentials=credentials,
    client_options={"quota_project_id": PROJECT_ID}
)

BATCH_SIZE = 100
batches = [to_process[i:i+BATCH_SIZE] for i in range(0, len(to_process), BATCH_SIZE)]
print(f"\nSubmitting {len(batches)} async OCR batches...")

operations = []
for batch_idx, batch in enumerate(tqdm(batches, desc="Submitting")):
    requests_list = []
    for pdf_path in batch:
        basename = os.path.basename(pdf_path)
        gcs_source = f"gs://{BUCKET_NAME}/{GCS_INPUT_PREFIX}{basename}"
        gcs_dest = f"gs://{BUCKET_NAME}/{GCS_OUTPUT_PREFIX}{os.path.splitext(basename)[0]}/"
        input_config = vision.InputConfig(
            gcs_source=vision.GcsSource(uri=gcs_source),
            mime_type="application/pdf"
        )
        output_config = vision.OutputConfig(
            gcs_destination=vision.GcsDestination(uri=gcs_dest),
            batch_size=50
        )
        request = vision.AsyncAnnotateFileRequest(
            features=[vision.Feature(type_=vision.Feature.Type.DOCUMENT_TEXT_DETECTION)],
            input_config=input_config,
            output_config=output_config,
        )
        requests_list.append(request)
    operation = vision_client.async_batch_annotate_files(requests=requests_list)
    operations.append((batch_idx, operation))
    print(f"  Batch {batch_idx + 1}/{len(batches)} submitted ({len(batch)} files)")

# --- Step 3: Wait for completion ---
print(f"\nWaiting for {len(operations)} operations to complete...")
for batch_idx, operation in tqdm(operations, desc="Waiting"):
    result = operation.result(timeout=3600)
    print(f"  Batch {batch_idx + 1} complete")

# --- Step 4: Download results ---
print(f"\nDownloading results from GCS...")
downloaded = 0
for pdf_path in tqdm(to_process, desc="Downloading"):
    basename = os.path.splitext(os.path.basename(pdf_path))[0]
    out_path = os.path.join(GCV_OUT_DIR, f"{basename}.md")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        downloaded += 1; continue
    prefix = f"{GCS_OUTPUT_PREFIX}{basename}/"
    blobs = list(bucket.list_blobs(prefix=prefix))
    if not blobs: continue
    all_text = []
    for blob in sorted(blobs, key=lambda b: b.name):
        if not blob.name.endswith('.json'): continue
        content = json.loads(blob.download_as_string())
        for response in content.get("responses", []):
            annotation = response.get("fullTextAnnotation", {})
            text = annotation.get("text", "")
            if text: all_text.append(text)
    if all_text:
        with open(out_path, 'w', encoding='utf-8') as f:
            f.write("\n\n---\n\n".join(all_text))
        downloaded += 1

print(f"\nDone: {downloaded} files downloaded to {GCV_OUT_DIR}")
print(f"Total corpus: {skipped + downloaded} files ready")


## Cell 8: Clean OCR Output

Strips form boilerplate, OCR noise, table artifacts, classification stamps, Hynek index duplicates, and image placeholders from both Marker and GCV output. Filters out files with <100 chars or >15% non-ASCII (Unicode garbage from bad embedded text).

In [ ]:
import re
from tqdm import tqdm

def clean_text(text):
    text = re.sub(r'\{[\d]+\}-+', '\n', text)
    text = re.sub(r'!\[\]\([^)]+\)', '', text)
    text = re.sub(r'^\s*\|[-=\s|:]+\|\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*\|[\s|]*\|\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'^\s*(UN)?CLASSIFIE?D?\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*RESTRICTED\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*CONFIDENTIAL\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*SECRET\s*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*ATIC FORM 329.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*FTD SEP 63.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'^\s*AF FORM 112.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'Previous editions of this form may be used\.?', '', text, flags=re.IGNORECASE)
    text = re.sub(r'U\.\s*S\.\s*GOVERNMENT PRINTING OFFICE.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'Dr\.?\s*HYNEK.S\s+EVALUATIONS\s+EXTRACTED\s+FROM\s+PROJECT\s+GRUDGE\s+REPORT\.?.*?(?=\n\n[A-Z]|\Z)', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'INCIDENT\s+INDEX\s*\n.*?(?:Astronomical|Non-astronomical).*?(?=\n\n[A-Z]|\Z)', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'(THE RESIDENCE OF[A-Z\s,]+){2,}', '', text)
    text = re.sub(r'(THE PERSON OF[A-Z\s,]+){2,}', '', text)
    text = re.sub(r'(of the Control of[a-zA-Z\s,]+){2,}', '', text)
    text = re.sub(r'(of the same of[a-zA-Z\s,]+){2,}', '', text)
    text = re.sub(r'(the service of[a-zA-Z\s,]+){2,}', '', text)
    text = re.sub(r'^\s*[-=\.\*\s]{10,}\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'DOWNGRADED.*?DECLASSIFIED.*$', '', text, flags=re.MULTILINE | re.IGNORECASE)
    text = re.sub(r'NOTE:\s*THIS DOCUMENT CONTAINS INFORMATION AFFECTING THE NATIONAL DEFENSE.*?DIRECTOR OF INTELLIGENCE,\s*USAF\.?', '', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'^\s*\|\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\s*\|\s*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n{4,}', '\n\n\n', text)
    return text.strip()

def is_readable(text):
    data = text.encode('utf-8')
    total = len(data)
    if total == 0: return False
    ascii_count = sum(1 for b in data if 32 <= b <= 126 or b in (10, 13, 9))
    return (ascii_count / total) > 0.85

source_dirs = [
    (MARKER_OUT_DIR, "marker"),
    (GCV_OUT_DIR, "gcv"),
]

total = 0; cleaned = 0; too_short = 0; garbage = 0

for src_dir, label in source_dirs:
    if not os.path.exists(src_dir): continue
    md_files = [f for f in os.listdir(src_dir) if f.endswith('.md')]
    print(f"Processing {len(md_files)} files from {label}...")
    for md_file in tqdm(md_files, desc=f"Cleaning {label}"):
        total += 1
        dst_path = os.path.join(CLEAN_DIR, md_file)
        if os.path.exists(dst_path): cleaned += 1; continue
        with open(os.path.join(src_dir, md_file), 'r', encoding='utf-8') as f:
            raw = f.read()
        clean = clean_text(raw)
        if len(clean) < 100: too_short += 1; continue
        if not is_readable(clean): garbage += 1; continue
        with open(dst_path, 'w', encoding='utf-8') as f:
            f.write(clean)
        cleaned += 1

print(f"\nTotal: {total}, Cleaned: {cleaned}, Too short: {too_short}, Garbage: {garbage}")
print(f"Files in {CLEAN_DIR}: {len([f for f in os.listdir(CLEAN_DIR) if f.endswith('.md')])}")

## Cell 9: Domain-Aware Spell Correction

Corrects OCR misspellings while protecting military ranks, aircraft designators, base names, project terms, personnel names, and anything with digits or abbreviation patterns.

In [ ]:
import re
from tqdm import tqdm
from symspellpy import SymSpell, Verbosity

sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
import pkg_resources
dict_path = pkg_resources.resource_filename("symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dict_path, term_index=0, count_index=1)
print(f"Dictionary: {sym_spell.word_count} words")

PROTECTED = set()
for terms in [
    {"pvt","pfc","cpl","sgt","ssgt","tsgt","msgt","lt","1lt","2lt","capt","maj","col","ltcol","gen"},
    {"p-80","p-51","f-51","f-94","f-94b","f-86","c-47","c-54","b-24","b-29","b-52","dc-3","rf-80","t-33"},
    {"afb","usaf","usaaf","fbi","cia","oni","osi","atic","amc","gci","ufo","ufos"},
    {"wright-patterson","godman","hamilton","edwards","dover","selfridge","kirtland","holloman","robins"},
    {"hynek","ruppelt","arnold","mantell","mccoy","clingerman","sanderson","nunley","lemon"},
    {"bluebook","grudge","sign","mogul","flyobrpt","radiosonde","mph","knots","ft","est","cst","pst"},
]:
    for t in terms:
        PROTECTED.add(t.lower()); PROTECTED.add(t.upper()); PROTECTED.add(t.title())

PROTECTED_PATTERNS = [r'\d', r'^[A-Z]{2,}$', r'[-/]', r'°']

def is_protected(word):
    clean = word.strip(".,;:!?\"'()[]")
    if clean.lower() in PROTECTED or len(clean) <= 2: return True
    return any(re.search(p, clean) for p in PROTECTED_PATTERNS)

def correct_text(text):
    words = text.split()
    corrected = []; fixes = 0
    for word in words:
        if is_protected(word): corrected.append(word); continue
        prefix = suffix = ""
        clean = word
        while clean and clean[0] in '("\'[': prefix += clean[0]; clean = clean[1:]
        while clean and clean[-1] in '.,;:!?"\')]': suffix = clean[-1] + suffix; clean = clean[:-1]
        if not clean or is_protected(clean): corrected.append(word); continue
        suggestions = sym_spell.lookup(clean, Verbosity.CLOSEST, max_edit_distance=2)
        if suggestions and suggestions[0].term != clean.lower() and suggestions[0].distance > 0 and suggestions[0].distance <= 2:
            repl = suggestions[0].term
            if clean[0].isupper() and len(clean) > 1 and clean[1:].islower(): repl = repl.capitalize()
            elif clean.isupper(): repl = repl.upper()
            corrected.append(prefix + repl + suffix); fixes += 1
        else: corrected.append(word)
    return ' '.join(corrected), fixes

md_files = [f for f in os.listdir(CLEAN_DIR) if f.endswith('.md')]
print(f"Spell-correcting {len(md_files)} files...")
total_fixes = 0
for md_file in tqdm(md_files, desc="Correcting"):
    dst = os.path.join(CORRECTED_DIR, md_file)
    if os.path.exists(dst): continue
    with open(os.path.join(CLEAN_DIR, md_file), 'r', encoding='utf-8') as f: text = f.read()
    corrected, fixes = correct_text(text)
    total_fixes += fixes
    with open(dst, 'w', encoding='utf-8') as f: f.write(corrected)

print(f"\nTotal corrections: {total_fixes}")
print(f"Average per file: {total_fixes / max(len(md_files), 1):.1f}")

## Cell 10: Full Corpus Embedding + Clustering

**Input:** `corpus_corrected/` (spell-corrected, cleaned OCR text)

**Steps:**
1. Build case index from corrected text, parsing date/location/case ID from filenames
2. Embed full documents with `nomic-embed-text-v1.5` (8192 token context — no truncation)
3. UMAP dimensionality reduction + HDBSCAN clustering
4. Visualization + cluster summaries

**Output:** Parquet index, embeddings (.npy), UMAP coords, cluster labels, visualization, JSON results.

In [ ]:
import os, re, json
import numpy as np
import polars as pl
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import torch

# --- Step 1: Index ---
print("Building case index...")
all_cases = []
for md_file in sorted(os.listdir(CORRECTED_DIR)):
    if not md_file.endswith('.md'): continue
    with open(os.path.join(CORRECTED_DIR, md_file), 'r', encoding='utf-8') as f: text = f.read()
    if len(text) < 100: continue
    basename = md_file.replace('.md', '')
    parts = basename.split('-', 3)
    year = parts[0] if len(parts) > 0 else "unknown"
    month = parts[1] if len(parts) > 1 else "unknown"
    case_id = parts[2] if len(parts) > 2 else basename
    location = parts[3].replace('-', ' ') if len(parts) > 3 else "unknown"
    decade = "unknown"
    if year.isdigit():
        yr = int(year)
        if 1940 <= yr < 1950: decade = "1940s"
        elif 1950 <= yr < 1960: decade = "1950s"
        elif 1960 <= yr < 1970: decade = "1960s"
        elif 1970 <= yr < 1980: decade = "1970s"
    all_cases.append({"case_id": case_id, "filename": basename, "year": year,
                      "month": month, "decade": decade, "location": location,
                      "text": text, "char_count": len(text)})

print(f"Indexed {len(all_cases)} cases")
case_df = pl.DataFrame(all_cases)
case_df.write_parquet(f"{FINAL_CORPUS_DIR}/blue_book_cases_full.parquet")
for decade in sorted(case_df["decade"].unique().to_list()):
    subset = case_df.filter(pl.col("decade") == decade)
    print(f"  {decade}: {len(subset)} cases, median {subset['char_count'].median():.0f} chars")

# --- Step 2: Embed ---
print("\nLoading nomic-embed-text-v1.5...")
model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
texts = [f"search_document: {c['text']}" for c in all_cases]
filenames = [c["filename"] for c in all_cases]
print(f"Encoding {len(texts)} full documents...")
embeddings = model.encode(texts, batch_size=16, show_progress_bar=True,
                          device="cuda" if torch.cuda.is_available() else "cpu")
np.save(f"{FINAL_CORPUS_DIR}/full_embeddings_nomic.npy", embeddings)
print(f"Embeddings: {embeddings.shape}")

# --- Step 3: UMAP + HDBSCAN ---
import umap, hdbscan
print("\nUMAP...")
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
coords_2d = reducer.fit_transform(embeddings)
print("HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean', cluster_selection_method='eom')
cluster_labels = clusterer.fit_predict(coords_2d)
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
noise_pct = (cluster_labels == -1).sum() / len(cluster_labels) * 100
print(f"Found {n_clusters} clusters ({noise_pct:.1f}% noise)")
np.save(f"{FINAL_CORPUS_DIR}/umap_coords.npy", coords_2d)
np.save(f"{FINAL_CORPUS_DIR}/cluster_labels.npy", cluster_labels)

# --- Step 4: Visualization ---
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
noise_mask = cluster_labels == -1
ax.scatter(coords_2d[noise_mask, 0], coords_2d[noise_mask, 1], c='lightgray', alpha=0.3, s=8)
cmap = plt.cm.tab20(np.linspace(0, 1, 20))
for i, cl in enumerate(sorted(set(cluster_labels) - {-1})):
    mask = cluster_labels == cl
    ax.scatter(coords_2d[mask, 0], coords_2d[mask, 1], c=[cmap[i % 20]], alpha=0.7, s=15)
ax.set_title(f"Project Blue Book — {len(all_cases)} Cases\n{n_clusters} clusters | {noise_pct:.0f}% noise", fontsize=14)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.savefig(f"{VIZ_DIR}/full_embedding_map.png", dpi=200)
plt.show()

# --- Step 5: Cluster summaries ---
print("\n" + "="*60)
for cl in sorted(set(cluster_labels)):
    if cl == -1: continue
    members = [filenames[i] for i in range(len(filenames)) if cluster_labels[i] == cl]
    if len(members) < 5: continue
    print(f"\nCluster {cl} ({len(members)} cases):")
    for m in members[:5]: print(f"  {m}")
    if len(members) > 5: print(f"  ... and {len(members) - 5} more")

# --- Step 6: Save ---
results = [{"filename": filenames[i], "x": float(coords_2d[i, 0]), "y": float(coords_2d[i, 1]),
            "cluster": int(cluster_labels[i]), "year": all_cases[i]["year"],
            "location": all_cases[i]["location"], "char_count": all_cases[i]["char_count"]}
           for i in range(len(all_cases))]
with open(f"{DATA_DIR}/full_cluster_results.json", "w") as f:
    json.dump(results, f, indent=2)

checkpoint = {"total_cases": len(all_cases), "n_clusters": n_clusters,
              "noise_pct": round(noise_pct, 1), "embedding_model": "nomic-ai/nomic-embed-text-v1.5",
              "embedding_dim": int(embeddings.shape[1]), "status": "complete"}
with open(f"{DATA_DIR}/checkpoint.json", "w") as f:
    json.dump(checkpoint, f, indent=2)
print(f"\nAll results saved. Checkpoint written.")

## Cell 11: BERTopic Topic Extraction

Runs BERTopic on pre-computed embeddings to extract keyword-based topic descriptions for each cluster. Installs its own dependencies. **Do NOT run Cell 2 first** (version conflicts). Restart runtime, run Cell 1, then this cell directly.


In [ ]:
!pip install bertopic sentence-transformers transformers -q

import os
import json
import numpy as np
from bertopic import BERTopic

PROJECT_DIR = "/content/drive/MyDrive/blue_book_phenomenology"
FINAL_CORPUS_DIR = f"{PROJECT_DIR}/corpus"
CORRECTED_DIR = f"{PROJECT_DIR}/corpus_corrected"
DATA_DIR = f"{PROJECT_DIR}/results/data"
MODELS_DIR = f"{PROJECT_DIR}/results/models"

# Load saved embeddings
embeddings = np.load(f"{FINAL_CORPUS_DIR}/full_embeddings_nomic.npy")

# Load texts
with open(f"{DATA_DIR}/full_cluster_results.json") as f:
    results = json.load(f)

texts = []
for r in results:
    fpath = os.path.join(CORRECTED_DIR, r["filename"] + ".md")
    if os.path.exists(fpath):
        with open(fpath, 'r') as f:
            texts.append(f.read()[:2000])
    else:
        texts.append("")

print(f"Loaded {len(texts)} texts, {embeddings.shape} embeddings")

# Run BERTopic fresh using pre-computed embeddings
from umap import UMAP
from hdbscan import HDBSCAN

umap_model = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=5, metric='euclidean', cluster_selection_method='eom')

topic_model = BERTopic(umap_model=umap_model, hdbscan_model=hdbscan_model, verbose=True)
topics, probs = topic_model.fit_transform(texts, embeddings)

# Print results
info = topic_model.get_topic_info()
print(f"\nTotal topics: {len(info)}")

for _, row in info.head(40).iterrows():
    if row['Topic'] == -1:
        print(f"\nTopic -1 (noise): {row['Count']} cases")
        continue
    topic = topic_model.get_topic(row['Topic'])
    keywords = ", ".join([w for w, _ in topic[:10]])
    print(f"\nTopic {row['Topic']} ({row['Count']} cases): {keywords}")

# Save fresh model
topic_model.save(f"{MODELS_DIR}/bertopic_model_v2")
print("\nModel saved.")
